# Woodbench Dry-Run Test

Test toan bo pipeline tren CPU (dry-run) truoc khi chay tren GPU thuc te.

**Muc tieu:**
- Xac nhan tat ca imports hoat dong
- Xac nhan model tao dung output shape
- Xac nhan optimizer phan biet backbone/head
- Xac nhan training loop chay duoc 1 epoch
- Xac nhan Grad-CAM tao duoc visualization

**Luu y:** Chay tren CPU, chi test logic, khong can GPU.
**Thoi gian uoc tinh:** 5-10 phut tren Colab CPU.

## 1. Setup — Clone repo & install dependencies

In [ ]:
# Clone repo neu can
!git clone https://github.com/ManifoldsTeam/woodbench.git 2>/dev/null || echo "Repo may already exist"
%cd woodbench

# Install dependencies
!pip install -q torch torchvision timm scikit-learn matplotlib seaborn pillow numpy pandas
!pip install -q hydra-core omegaconf torchcam thop 2>/dev/null || echo "torchcam/thop may already installed"

import sys
sys.path.insert(0, '/content/woodbench')
print("Setup complete")

## 2. Test Imports

In [ ]:
print("=== Test 2.1: Import all modules ===")

try:
    from src import (
        ConiferDataset, get_dataloaders,
        get_train_transforms, get_val_transforms,
        create_model, BACKBONE_NAMES,
        TrainingEngine,
        create_optimizer, create_scheduler, create_loss_function,
        LabelSmoothingCrossEntropy, FocalLoss,
        compute_metrics, measure_inference_time,
        plot_confusion_matrix, save_confusion_matrix,
        set_seed, setup_logger,
        ModelCheckpoint, load_checkpoint, save_checkpoint,
        create_stratified_split, verify_split,
        benchmark_all_models, compare_models,
    )
    print("[PASS] All imports successful")
    print(f"  BACKBONE_NAMES: {list(BACKBONE_NAMES.keys())}")
except ImportError as e:
    print(f"[FAIL] Import error: {e}")
    raise

## 3. Test Data — Create mock dataset & transforms

In [ ]:
import tempfile
from pathlib import Path
from PIL import Image
import numpy as np

print("=== Test 3.1: Create mock dataset structure ===")

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    
    # Create folder structure: data/{split}/{family}/{species}/*.jpg
    families = ["Cupressaceae", "Pinaceae"]
    species = {
        "Cupressaceae": ["cunninghamia_lanceolata", "fokienia_hodginsii"],
        "Pinaceae": ["pinus_dalatensis", "pinus_radiata"],
    }
    splits = ["train", "val", "test"]
    
    for split in splits:
        for family in families:
            for sp in species[family]:
                folder = tmp_path / split / family / sp
                folder.mkdir(parents=True, exist_ok=True)
                # Create 3 dummy images per species
                for i in range(3):
                    img = Image.new("RGB", (256, 256), color=(np.random.randint(0, 255), 100, 50))
                    img.save(folder / f"img_{i}.jpg")
    
    print(f"  Created mock dataset at {tmp_path}")
    
    print("\n=== Test 3.2: ConiferDataset — Task A ===")
    dataset = ConiferDataset(tmp_path, split="train", task="taskA")
    print(f"  [PASS] Dataset loaded: {len(dataset)} samples")
    print(f"  Classes: {dataset.classes}")
    print(f"  Class to idx: {dataset.class_to_idx}")
    
    img, label = dataset[0]
    print(f"  [PASS] __getitem__ returns tensor shape: {img.shape}, label: {label}")
    
    print("\n=== Test 3.3: ConiferDataset — Task B ===")
    dataset_b = ConiferDataset(tmp_path, split="train", task="taskB")
    print(f"  [PASS] Task B dataset: {len(dataset_b)} samples")
    print(f"  Task B classes: {dataset_b.classes}")
    
    print("\n=== Test 3.4: get_dataloaders ===")
    train_loader, val_loader, test_loader = get_dataloaders(tmp_path, batch_size=4)
    print(f"  [PASS] Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}, Test: {len(test_loader.dataset)}")
    print(f"  Train batches: {len(train_loader)}")
    
    print("\n=== Test 3.5: Transforms ===")
    train_tf = get_train_transforms(img_size=224, use_90rotation=False)
    val_tf = get_val_transforms(img_size=224)
    dummy_img = Image.new("RGB", (512, 512), color="red")
    out = val_tf(dummy_img)
    print(f"  [PASS] val_transform: {dummy_img.size} -> {out.shape}")
    
    # Test 90 rotation
    train_tf_90 = get_train_transforms(img_size=224, use_90rotation=True)
    out_90 = train_tf_90(dummy_img)
    print(f"  [PASS] 90rotation transform: {out_90.shape}")
    
    print("\n=== Test 3.6: verify_split ===")
    stats = verify_split(tmp_path)
    print(f"  [PASS] Split verification:")
    for split, info in stats.items():
        if isinstance(info, dict) and info.get("exists"):
            print(f"    {split}: {info['total']} samples")

## 4. Test Model Factory — All 5 backbones

In [ ]:
import torch

print("=== Test 4: Model Factory — all 5 backbones ===")

device = "cpu"
all_passed = True

for backbone in BACKBONE_NAMES.keys():
    try:
        model = create_model(backbone, num_classes=4, pretrained=False)
        model = model.to(device)
        
        # Test output shape
        dummy = torch.randn(2, 3, 224, 224).to(device)
        out = model(dummy)
        assert out.shape == (2, 4), f"Expected (2, 4), got {out.shape}"
        
        # Test Task B (20 classes)
        model_20 = create_model(backbone, num_classes=20, pretrained=False)
        out_20 = model_20(dummy)
        assert out_20.shape == (2, 20), f"Expected (2, 20), got {out_20.shape}"
        
        # Test eval mode
        model.eval()
        assert not model.training
        
        print(f"  [PASS] {backbone}: output={out.shape}, TaskB={out_20.shape}")
    except Exception as e:
        print(f"  [FAIL] {backbone}: {e}")
        all_passed = False

print(f"\n{'[ALL PASS]' if all_passed else '[SOME FAILED]'} Model factory tests")

## 5. Test Optimizer — Differential Learning Rates

In [ ]:
print("=== Test 5: Optimizer with differential LR ===")

model = create_model("resnet50", num_classes=4, pretrained=False, freeze_backbone=False)

# Test 1: Same LR for all params
opt_same = create_optimizer(model, lr=1e-4)
assert len(opt_same.param_groups) == 1, "Expected single param group"
print(f"  [PASS] Same LR: {len(opt_same.param_groups)} param group(s)")

# Test 2: Differential LR (backbone=1e-5, head=1e-4)
opt_diff = create_optimizer(model, lr=1e-4, backbone_lr=1e-5)
print(f"  Param groups: {len(opt_diff.param_groups)}")
for i, pg in enumerate(opt_diff.param_groups):
    print(f"    Group {i} ({pg['name']}): lr={pg['lr']:.0e}, n_params={len(pg['params'])}")

# Verify head has higher LR than backbone
lrs = {pg["name"]: pg["lr"] for pg in opt_diff.param_groups}
if "head" in lrs and "backbone" in lrs:
    if lrs["head"] > lrs["backbone"]:
        print(f"  [PASS] Head LR ({lrs['head']:.0e}) > Backbone LR ({lrs['backbone']:.0e})")
    else:
        print(f"  [WARN] Check LR assignment: {lrs}")
else:
    print(f"  [INFO] Param group names: {list(lrs.keys())}")

# Test 3: Frozen backbone (linear probing)
model_frozen = create_model("resnet50", num_classes=4, pretrained=False, freeze_backbone=True)
frozen_params = sum(1 for p in model_frozen.parameters() if not p.requires_grad)
trainable_params = sum(1 for p in model_frozen.parameters() if p.requires_grad)
print(f"  [PASS] Frozen backbone: {frozen_params} frozen, {trainable_params} trainable")

print("\n=== Test 5b: Loss Functions ===")

criterion = create_loss_function("label_smoothing", label_smoothing=0.1)
print(f"  [PASS] LabelSmoothingCrossEntropy: {type(criterion).__name__}")

preds = torch.randn(4, 4)
targets = torch.tensor([0, 1, 2, 3])
loss = criterion(preds, targets)
print(f"  [PASS] Forward pass: loss={loss.item():.4f}")

# Test FocalLoss
focal = create_loss_function("focal", alpha=1.0, gamma=2.0)
loss_focal = focal(preds, targets)
print(f"  [PASS] FocalLoss: {loss_focal.item():.4f}")

## 6. Test Training Engine — 1 epoch dry-run

In [ ]:
import torch
import torch.nn as nn

print("=== Test 6: Training Engine — 1 epoch dry-run ===")

device = "cpu"

# Create small model for fast testing
model = create_model("resnet50", num_classes=4, pretrained=False)

# Use mock dataloader (random data)
class RandomDataset(torch.utils.data.Dataset):
    def __init__(self, size=20):
        self.size = size
    def __len__(self):
        return self.size
    def __getitem__(self, idx):
        return torch.randn(3, 224, 224), torch.randint(0, 4, (1,)).item()

train_ds = RandomDataset(20)
val_ds = RandomDataset(10)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=4, shuffle=False)

optimizer = create_optimizer(model, lr=1e-4)
scheduler = create_scheduler(optimizer, epochs=10, scheduler_type="cosine")
criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

engine = TrainingEngine(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    use_amp=False,  # CPU, disable AMP
)

print(f"  Model device: {next(model.parameters()).device}")
print(f"  Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

print("\n  Training 1 epoch...")
train_metrics = engine.train_one_epoch()
print(f"  Train: loss={train_metrics['train_loss']:.4f}, acc={train_metrics['train_acc']:.2f}%")

val_metrics = engine.validate()
print(f"  Val:   loss={val_metrics['val_loss']:.4f}, acc={val_metrics['val_acc']:.2f}%")

print(f"\n  Scheduler: {type(scheduler).__name__}")
print(f"  Current LR: {optimizer.param_groups[0]['lr']:.2e}")

print("\n[PASS] Training engine dry-run complete")

## 7. Test Evaluation Metrics

In [ ]:
print("=== Test 7: Evaluation Metrics ===")

model = create_model("resnet50", num_classes=4, pretrained=False)
model.eval()

class RandomDataset(torch.utils.data.Dataset):
    def __init__(self, size=20, num_classes=4):
        self.size = size
        self.num_classes = num_classes
    def __len__(self):
        return self.size
    def __getitem__(self, idx):
        return torch.randn(3, 224, 224), torch.randint(0, self.num_classes, (1,)).item()

test_ds = RandomDataset(20, num_classes=4)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=4)

print("\n=== Test 7.1: compute_metrics ===")
metrics = compute_metrics(
    model=model,
    data_loader=test_loader,
    device="cpu",
    num_classes=4,
    class_names=["Cupressaceae", "Pinaceae", "Podocarpaceae", "Taxaceae"],
)
print(f"  Accuracy:      {metrics['accuracy']*100:.2f}%")
print(f"  Precision:     {metrics['precision_macro']*100:.2f}%")
print(f"  Recall:        {metrics['recall_macro']*100:.2f}%")
print(f"  F1-macro:      {metrics['f1_macro']*100:.2f}%")
print(f"  Confusion matrix shape: {metrics['confusion_matrix'].shape}")
print(f"  Per-class: {list(metrics['per_class'].keys())}")
print("  [PASS] compute_metrics works")

print("\n=== Test 7.2: measure_inference_time ===")
timing = measure_inference_time(
    model=model,
    input_size=(3, 224, 224),
    device="cpu",
    num_runs=20,
    batch_size=1,
)
print(f"  Mean: {timing['mean_ms']:.2f} ms/image")
print(f"  Std:  {timing['std_ms']:.2f} ms")
print("  [PASS] measure_inference_time works")

## 8. Test Grad-CAM

In [ ]:
print("=== Test 8: Grad-CAM with torchcam ===")

try:
    from src.eval.gradcam import GradCAMGenerator
    
    model = create_model("resnet50", num_classes=4, pretrained=False)
    model.eval()
    
    cam_gen = GradCAMGenerator(model, backbone_name="resnet50", method="layer")
    print(f"  [PASS] GradCAMGenerator created for resnet50")
    print(f"  Target layer: {cam_gen.target_layer}")
    
    # Generate CAM on dummy image
    dummy_img = torch.randn(3, 224, 224)
    cam_vis = cam_gen.generate(dummy_img)
    print(f"  [PASS] CAM generated: shape={cam_vis.shape}")
    
    # Test other backbones
    for backbone in ["efficientnet_b3", "convnext_tiny"]:
        try:
            m = create_model(backbone, num_classes=4, pretrained=False)
            cam = GradCAMGenerator(m, backbone_name=backbone)
            out = cam.generate(dummy_img)
            print(f"  [PASS] {backbone}: CAM shape={out.shape}")
        except Exception as e:
            print(f"  [WARN] {backbone}: {e}")
    
    print("\n[PASS] Grad-CAM tests complete")
    
except ImportError as e:
    print(f"[SKIP] torchcam not available: {e}")
    print("  Run: pip install torchcam")

## 9. Test Checkpointing

In [ ]:
import tempfile
from pathlib import Path

print("=== Test 9: Checkpoint Save/Load ===")

model = create_model("resnet50", num_classes=4, pretrained=False)
optimizer = create_optimizer(model, lr=1e-4)

with tempfile.TemporaryDirectory() as tmp:
    ckpt_path = Path(tmp) / "test_checkpoint.pth"
    
    save_checkpoint(
        checkpoint_path=ckpt_path,
        model=model,
        optimizer=optimizer,
        epoch=9,
        metrics={"train_loss": 0.5, "val_acc": 95.0},
        best_val_acc=95.0,
        best_epoch=7,
    )
    print(f"  [PASS] Checkpoint saved: {ckpt_path} ({ckpt_path.stat().st_size} bytes)")
    
    # Load into new model
    model2 = create_model("resnet50", num_classes=4, pretrained=False)
    opt2 = create_optimizer(model2, lr=1e-4)
    
    info = load_checkpoint(ckpt_path, model2, opt2, device="cpu")
    print(f"  [PASS] Checkpoint loaded: epoch={info['epoch']}, best_val_acc={info['best_val_acc']}")
    print(f"  Metrics: {info['metrics']}")
    
    # Verify weights match
    for (n1, p1), (n2, p2) in zip(model.named_parameters(), model2.named_parameters()):
        assert torch.allclose(p1, p2), f"Weight mismatch: {n1}"
    print("  [PASS] Weights match after load")

## 10. Summary

In [ ]:
print("=" * 60)
print("WOODBENCH DRY-RUN TEST SUMMARY")
print("=" * 60)
print(""
All tests passed. Pipeline is ready for GPU training.

Next steps:
1. Upload dataset to Colab or mount Google Drive
2. Run full training: python train.py experiment=taskA model.backbone=resnet50
3. Run multi-model: python train.py --multirun model.backbone=[resnet50,efficientnet_b3]
4. Evaluate: python eval.py checkpoint=/path/to/model_best.pth
5. Profile: python profile.py model.backbone=resnet50

Note: Dry-run tested on CPU. GPU training requires CUDA.
""")
print("=" * 60)